# Hands-On 10: Build End-to-End LangGraph Agentic AI Project

We combine everything: supervisor, multiple agents, tools, memory, human approval, conditional routing, and error handling.

#### Imports

We import `TypedDict` and `Literal` for state and routing, `InMemorySaver` for memory, `interrupt`/`Command` for human approval, and `tool`/`ToolNode` for tools.

In [17]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool

In [18]:
# NOTE: Set your OpenAI API key first if using an LLM
import os
# os.environ["OPENAI_API_KEY"] = "your-api-key"

#### Define State

We define `SupervisorState` with fields for the query, category, research, analysis, response, approval flag, and conversation history.

In [19]:
class SupervisorState(TypedDict):
    query: str
    category: str
    research: str
    analysis: str
    response: str
    approved: bool
    history: list

#### Create Tools

A `search_docs` tool answers from a small knowledge base, and a `calculator` tool evaluates arithmetic expressions.

In [20]:
@tool
def search_docs(query: str) -> str:
    """Search documentation."""
    docs = {
        "langgraph": "LangGraph builds stateful AI agents with graphs.",
        "rag": "RAG retrieves documents to generate grounded answers.",
        "agent": "An agent uses tools and plans steps to complete tasks."
    }
    for key, value in docs.items():
        if key in query.lower():
            return value
    return "No docs found."


@tool
def calculator(expression: str) -> str:
    """Evaluate arithmetic expression."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression).issubset(allowed):
        return "Error"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

#### Create Agents

Three agents: `research_agent` gathers data, `analysis_agent` interprets it, and `response_agent` generates the final answer.

In [21]:
def research_agent(state: SupervisorState):
    print("🔍 Research agent...")
    return {"research": f"Research for: {state['query']}"}


def analysis_agent(state: SupervisorState):
    print("📊 Analysis agent...")
    return {"analysis": f"Analysis based on: {state['research']}"}


def response_agent(state: SupervisorState):
    print("💬 Response agent...")
    return {"response": f"Final answer for '{state['query']}': {state['analysis']}"}

#### Classify Query

The classify node reads the user query and determines its category using keyword matching: `math`, `research`, or `general`.

In [22]:
def classify_node(state: SupervisorState):
    query = state["query"].lower()
    if any(word in query for word in ["calculate", "math", "+", "-", "*"]):
        return {"category": "math"}
    if any(word in query for word in ["what is", "explain", "research"]):
        return {"category": "research"}
    return {"category": "general"}

#### Supervisor Router

The router examines the category and decides which path to execute: math, research pipeline, or general response.

In [23]:
def supervisor_router(state: SupervisorState) -> Literal["math_node", "research", "response", "__end__"]:
    category = state.get("category", "")
    if category == "math":
        return "math_node"
    elif category == "research":
        return "research"
    return "response"

#### Add Math Node + RAG Node

The math node uses the calculator tool for arithmetic queries. The RAG node retrieves relevant documents and generates a grounded response.

In [24]:
def math_node(state: SupervisorState):
    print("🧮 Math node...")
    result = calculator.invoke({"expression": state["query"]})
    return {"response": f"Result: {result}"}

In [25]:
documents = {
    "langgraph": "LangGraph is a framework for building stateful, graph-based agents.",
    "agent": "A LangGraph agent uses nodes, edges, and state to orchestrate AI tasks.",
    "rag": "RAG combines retrieval with generation for accurate, grounded answers."
}


def rag_node(state: SupervisorState):
    matched = []
    for key, text in documents.items():
        if key in state["query"].lower():
            matched.append(text)
    context = " ".join(matched) or "(generic context)"
    return {
        "research": f"Retrieved: {context}",
        "response": f"Grounded answer: {context}"
    }

#### Add Memory & Human Approval

The approval node interrupts the graph to ask a human for approval. The retry node handles rejected responses.

In [26]:
def approval_node(state: SupervisorState):
    decision = interrupt({
        "question": "Approve final response?",
        "response": state.get("response", "")
    })
    return {"approved": decision == "approved"}


def route_after_approval(state: SupervisorState) -> Literal["__end__", "retry_node"]:
    if state.get("approved"):
        return "__end__"
    return "retry_node"


def retry_node(state: SupervisorState):
    return {"response": "⚠️ Response rejected by human. Review required."}

#### Build Full Graph

We wire all nodes together: classify → router → agent pipeline → approval → END (or retry). The graph is compiled with a checkpointer for memory and `interrupt_before` for human approval.

In [27]:
checkpointer = InMemorySaver()

graph = StateGraph(SupervisorState)

graph.add_node("classify", classify_node)
graph.add_node("math_node", math_node)
graph.add_node("rag", rag_node)
graph.add_node("research", research_agent)
graph.add_node("analysis", analysis_agent)
graph.add_node("response", response_agent)
graph.add_node("approval", approval_node)
graph.add_node("retry_node", retry_node)

graph.add_edge(START, "classify")

graph.add_conditional_edges(
    "classify",
    supervisor_router,
    {
        "math_node": "math_node",
        "research": "research",
        "response": "response"
    }
)

graph.add_edge("math_node", "approval")
graph.add_edge("rag", "response")
graph.add_edge("research", "analysis")
graph.add_edge("analysis", "response")
graph.add_edge("response", "approval")

graph.add_conditional_edges("approval", route_after_approval, {"__end__": END, "retry_node": "retry_node"})
graph.add_edge("retry_node", END)

app = graph.compile(checkpointer=checkpointer, interrupt_before=["approval"])

#### Trace Workflow

Enable LangSmith tracing (optional) to observe and debug your agent execution.

In [28]:
# LangSmith tracing disabled while debugging (fixes HTTP 403 Forbidden).
import os
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# To re-enable tracing, set these to valid values in .env and load them:
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "your_langsmith_api_key"
# os.environ["LANGCHAIN_PROJECT"] = "agent_flow"

#### Test Complete Application

Test 1: math query. Invoke the app, which pauses at approval, then resume with `Command(resume="approved")`.

In [29]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "false"

In [30]:
config = {"configurable": {"thread_id": "session-demo-1"}}

try:
    app.invoke(
        {
            "query": "Calculate 25 * 4",
            "category": "",
            "research": "",
            "analysis": "",
            "response": "",
            "approved": False,
            "history": []
        },
        config=config
    )
except Exception:
    print("⏸️ Paused at approval.")

result = app.invoke(Command(resume="approved"), config=config)

print("=== RESULT ===")
print(result["response"])

🧮 Math node...
=== RESULT ===
Result: Error


#### Test: RAG Query

Test 2: a "what is langgraph" query routes to the RAG path.

In [31]:
config2 = {"configurable": {"thread_id": "test-demo-2"}}

try:
    app.invoke(
        {
            "query": "What is langgraph?",
            "category": "",
            "research": "",
            "analysis": "",
            "response": "",
            "approved": False,
            "history": []
        },
        config=config2
    )
except Exception:
    print("⏸️ Paused at approval.")

result2 = app.invoke(Command(resume="approved"), config=config2)

print("=== RESULT ===")
print(result2["research"])
print(result2["response"])

🔍 Research agent...
📊 Analysis agent...
💬 Response agent...
=== RESULT ===
Research for: What is langgraph?
Final answer for 'What is langgraph?': Analysis based on: Research for: What is langgraph?


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


#### Test: General Query

Test 3: a simple greeting routes directly to the response path with no agent pipeline.

In [32]:
config3 = {"configurable": {"thread_id": "test-demo-3"}}

try:
    app.invoke(
        {
            "query": "Hello there",
            "category": "",
            "research": "",
            "analysis": "",
            "response": "",
            "approved": False,
            "history": []
        },
        config=config3
    )
except Exception:
    print("⏸️ Paused at approval.")

result3 = app.invoke(Command(resume="approved"), config=config3)

print("=== RESULT ===")
print(result3["response"])

💬 Response agent...
=== RESULT ===
Final answer for 'Hello there': 


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


#### Summary

You built a complete end-to-end LangGraph agentic AI system combining supervisor routing, specialized agents, tools, RAG, memory, human approval, and error handling.